### Buy the stocks with the worst previous one-day returns, 
### and short the ones with the best previous one-day returns.

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

In [2]:
START_DATE = "2025-01-01"
END_DATE   = "2025-12-31"
DOWNLOAD_START = "2024-01-01"   # 多下一年用于 lag
ONE_WAY_TCOST = 0.0005
TRADING_DAYS = 252

In [4]:
symbols_url = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv"
symbols_df = pd.read_csv(symbols_url)
symbols = symbols_df["Symbol"].dropna().tolist()
# yfinance 用 '-' 代替 '.'
symbols = [s.replace(".", "-") for s in symbols]
len(symbols)

503

In [5]:
def download_prices(symbols, start, end, batch_size=50):
    all_close = []

    for i in range(0, len(symbols), batch_size):
        batch = symbols[i:i+batch_size]

        try:
            data = yf.download(
                batch,
                start=start,
                end=end,
                auto_adjust=True,
                progress=False,
                threads=False
            )

            if "Close" in data:
                all_close.append(data["Close"])

        except Exception as e:
            print(f"Batch {i} failed:", e)

    return pd.concat(all_close, axis=1)

cl = download_prices(
    symbols,
    start=DOWNLOAD_START,
    end=END_DATE
)

print("Downloaded stocks:", cl.shape[1])

HTTP Error 404: 

1 Failed download:
['WBA']: YFTzMissingError('possibly delisted; no timezone found')


Downloaded stocks: 503


In [33]:
# =========================
# 4. 数据清洗
# =========================
# 删除整列全 NaN 的股票
cl = cl.dropna(axis=1, how="all")

print("Any all-NaN columns left:", cl.isna().all().any())

Any all-NaN columns left: False


In [60]:
# =========================
# 5. 日收益率
# =========================
daily_ret = cl.pct_change(fill_method=None)

daily_ret_test = daily_ret.iloc[1:, :]
daily_ret_test.loc[:, daily_ret_test.isna().any(axis=0)] #这些是出现的晚或者中途退市的股票

Ticker,GEV,IPG,K,SOLV
Date,,,,
2024-01-03,NaN,-0.029607,-0.004848,NaN
2024-01-04,NaN,0.002179,-0.011134,NaN
2024-01-05,NaN,0.007766,-0.003167,NaN
2024-01-08,NaN,0.024353,-0.002118,NaN
2024-01-09,NaN,-0.009630,-0.002653,NaN
...,...,...,...,...
2025-12-23,-0.000544,NaN,NaN,-0.002847
2025-12-24,0.008875,NaN,NaN,-0.004593
2025-12-26,-0.005784,NaN,NaN,0.003492


In [61]:
# =========================
# 6. 市场等权收益（cross-sectional mean）
# =========================
market_daily_ret = daily_ret.mean(axis=1, skipna=True) #只要这天有一只股票存在，market_ret就不是nan
market_daily_ret

Date
2024-01-02         NaN
2024-01-03   -0.013398
2024-01-04   -0.001054
2024-01-05    0.002310
2024-01-08    0.011305
                ...   
2025-12-23   -0.002523
2025-12-24    0.003857
2025-12-26    0.000248
2025-12-29   -0.001669
2025-12-30   -0.001446
Length: 501, dtype: float64

In [40]:
# =========================
# 7. 构造均值回归权重（市场中性）
# =========================
# 每天可交易股票数量
valid_counts = cl.notna().sum(axis=1)
valid_counts

Date
2024-01-02    500
2024-01-03    500
2024-01-04    500
2024-01-05    500
2024-01-08    500
             ... 
2025-12-23    500
2025-12-24    500
2025-12-26    500
2025-12-29    500
2025-12-30    500
Length: 501, dtype: int64

In [77]:
# 均值回归：做空强的，做多弱的
weights = -(
    daily_ret.sub(market_daily_ret, axis=0)
).div(valid_counts, axis=0)

# 不可交易日权重清零（停牌 / 首日）
mask = cl.isna() | cl.shift(1).isna()
weights[mask] = 0.0

# 这些位置的weights本来是nan，被强制置0了，就是之前打印的出现晚或者退市了的四只股票，所以在True
# 的时候，这些股票在那天是不可交易的
mask.loc[:, mask.sum(axis=0)!=1] 

Ticker,GEV,IPG,K,SOLV
Date,,,,
2024-01-02,True,True,True,True
2024-01-03,True,False,False,True
2024-01-04,True,False,False,True
2024-01-05,True,False,False,True
2024-01-08,True,False,False,True
...,...,...,...,...
2025-12-23,False,True,True,False
2025-12-24,False,True,True,False
2025-12-26,False,True,True,False


In [78]:
# =========================
# 8. 每日 PnL（无交易成本）
# =========================
# 用 t-1 的权重吃 t 的收益
daily_pnl = (weights.shift(1) * daily_ret).sum(axis=1)
daily_pnl

Date
2024-01-02    0.000000e+00
2024-01-03    0.000000e+00
2024-01-04    4.623531e-05
2024-01-05   -2.026559e-05
2024-01-08    9.418241e-06
                  ...     
2025-12-23    8.282727e-06
2025-12-24    1.317347e-05
2025-12-26   -3.759871e-06
2025-12-29    1.059576e-07
2025-12-30   -1.604203e-05
Length: 501, dtype: float64

In [79]:
# =========================
# 9. 回测区间
# =========================
daily_pnl_2025 = daily_pnl.loc[START_DATE:END_DATE]
weights_2025   = weights.loc[START_DATE:END_DATE]

In [80]:
# =========================
# 10. Sharpe（无手续费）
# =========================
sharpe_no_tc = (
    np.sqrt(TRADING_DAYS)
    * daily_pnl_2025.mean()
    / daily_pnl_2025.std()
)

print("Sharpe (no transaction cost):", sharpe_no_tc)

Sharpe (no transaction cost): 1.223288475986047


In [81]:
# =========================
# 11. 加入交易成本
# =========================
# 日换手率
turnover = (weights_2025 - weights_2025.shift(1)).abs().sum(axis=1)

daily_pnl_tc = daily_pnl_2025 - turnover * ONE_WAY_TCOST

# =========================
# 12. Sharpe（含手续费）
# =========================
sharpe_with_tc = (
    np.sqrt(TRADING_DAYS)
    * daily_pnl_tc.mean()
    / daily_pnl_tc.std()
)

print("Sharpe (with transaction cost):", sharpe_with_tc)

Sharpe (with transaction cost): -0.07132280135613822
